DATA COLLECTION E PREPROCESSING

Cosa succede se alimentiamo un motore da formula uno con del fango invece che con benzina?
potete avere il modello più costoso, se i dati sono sporchi il risultato sarà inutile

Data collection e preprocessing sono le fasi con cui raccogli i dati e li prepari per il machine learning. Sono decisive: se i dati rappresentano male il problema, un modello sofisticato non risolve il difetto.

- ETL Pipelines: flussi di estrazione, trsformazione e caricamento dati
- Data Versioning (DVC): gestire i dataset con la stessa filosofia del codice
- Gestione formati: efifcienza computazionale co JSONL, Parquet e Arrow. Abbandonare i vecchi CSV a favore di nuove tecnologie.

Come si costruisce un flusso di dati indistruttibile?

Archiretture ETL per il Deep Learning
Costruire fondamenta solide per il modello.
Un sistema di intelligenza artificiale è limitato dalla qualità dei dati che riceve. Le pipeline ETL (extract, transform, load) automatizzano il passaggio del dato grezzo al dato pronto per l'inferenza.
Non ci limitiamo a spostare file, ma applichiamo logiche di pulizia, deduplicazione e normalizzazione sistematiche.
Questo flusso va automatizzato affinche il modello riceva sempre dati freschi e pronti

Il flusso dei dati
Dall'estrazione alla persistenza.
Il flusso si divide in 3 (ETL)
- Extraction: recupero multi-sorgente da API REST, database relazionali o scraping di documenti non strutturati.
- Transform: applicazione di filtri per la rimozione di valori anomali, gestione dei dati mancanti e conversione dei tipi
- Loading: caricamento dei dati trasformati in una 'Staging Area' o in un database vettoriale per il consumo immediato.
La qualità finale del dataset può essere espressa come il completamento del rapporto tra record errati e record totali.

* Raccogliere i dati
La data collection è l'acquisizione dei dati necessari al problema che si vuole affrontare. Le fonti possono essere database, sensori API, immagini o documenti.
Prima di estrarre devi definire:
- Cosa vuoi prevedere? - Per quale unità? - Quando fai le previoni? - Quali informazioni saranno disponibili in quel momento? - Quato storico serve?
Estrarre tutto e poi si vede, non è un buon punto di partenza. Si rischia di raccogliere molte colonne senza avere l'informazione necessaria.


Ma la pulizia non è solo togliere la polve

Il preprocessing è l'insieme delle trasformazioni che portano i dati raccolti a una forma coerente e utilizzabile dal modello.
Non significa solo eliminare i valori null, altre attività sono: - correggere i tipi - uniformare le unità - gestire dati mancanti - gestire anomalie - aggregare - codificare categorie (es. trasformare le famiglie articoli in variabili numeriche) - scalare valori (standarizzare alcune variabili numeriche) - Costruire variabili (per creare informazioni utili alla previsione), chiamato feature engineering.

Automazione e Pulizia
- Data Scrubbing: identificazione di rumore statistico e rimozione di informazioni sensibili per garantire la conformità legale ed etica.
- Schema Validation: verifica rigorosa che i dati estratti rispettino la struttura attesa per evitare crash improvvisi durante la fase di training.
- Incremental Loading: tecnica per processare solo i nuovi dati arrivati dall'ultima esecuzione, riducendo drasticamente il tempo di elaborazione.

Alcune trasformazioni dipendono dal modello scelto. 

Come facciamo a sapere se i dati che entrano oggi sono diversi da quelli di ieri?
Qui entra in giorco la statistica avanzata.
Le pipeline ETL devono integrare sistemi di monitoraggio per rilevare se la distribuzione dei dati in ingresso sta cambiando rispetto al passato.
Se la distanza è troppa, il modello deve essere riaddestrato.
L'allineamento statistico tra training e produzione è garantito dal monitoraggio continuo della densità di probabilità

Gestire la qualità è fondamentale, ma come facciamo a gestire i cambiamenti nel tempo?

Data Versioning con DVC
Trattare i dati come codice
DVC (Data Version Control): serve a gestire le versioni di dataset e modelli, collegandole ale versioni del codice.
Salvare il codice non basta per ricostruire un esperimento di machine learining. Devi sapere quali dati hai utilizzato.
Se sovrascrivi ogni volta i dati, perdi la possinilità di ricostuire correttamente le prove.
Con Git e DVC puoi conservare l'associazione tra versione del codice e versione dei dati, senza inserire il dataseteì completo nel repository Git.
Git non è adatto a gestire file binari da gigabyte. DVC (Data Version Control) risolve questo problema creando un ponte tra il codidce e lo storage remoto de dait.
Creaiamo dei piccoli file di testo che puntano ai dati reali salvati nel cloud.
Il file di puntamento dvc garantisce che ogni versione del modello sia legata in modo univoco alla versione del dataset corrispondente.

Come funziona questa sincronizzazione

Meccanica del versionamento
- DVC calcola un hash univoco del file e lo memorizza in un file di testo leggero tracciabile da Git.
- Remote Storage: i file pesanti vengono salvati su S3, Azuere o server locali, lasciando il repository snello.
- Pipeline DAG: DVC traccia le dipendenze tra script di preprocessing e file prodotti, rieseguendo solo ciò che è cambiato
- L'identità di un esperimento è data dalla combinazione dello stato del codice e dello stato dei dati versionati.

Git conserva codice, configurazioni e piccoli file di metadati DVC
Cache DVC conserva contenuto dei dati e dei modelli tracciati
Quando tracci i dati, DVC genera un piccolo file con le informazioni che identificano quel contenuto, incluso hash. Git conserva il riferimento, i dati rimangono separati.
Puoi configurare lo starage anche su file system locale, on infrastruttura interna. DVC non richiede di caricare i dati nel cloud. 

In questo tempo è possibile viaggiare tra i rami del tempo, è possibile tornare indietro alla versione di 3 mesi fa, recuperando codice e dati.


Archiviati i dati ora dobbiamo preoccuparci di quanto velocemente riusciamo a leggerli
I file csv sono lenti e pesanti.
File parquet o arrow sono più veloci.

In  un file 'normale' leggiamo tutta la riga anche se ci serve una colonna, Parquet le colonne sono separate, il modello può leggere solo le colonne interessate.
- JSONL: formato basato su righe, ideale per lo straming di testi, dove ogni riga è un oggetto JSON indipendente.
- Apache Parquet: formato colonnare compresso che permette di leggere solo le colonne necessarie al modello, risparmiando I/O
- Apache Arrow: formato in-memory che elimina il costo della serializzazione tra linguaggi (es. tra C++ e Python)
 il tempo di caricamento è ridotto grazie al memory-mapping che evita la copia fisica dei dati nella heap.

Oltre alla velocità guadagnamo in termini di spazio occupato.

Ottimizzazione computazionale
- Columnar Compression: Parquet utilizza algoritmi come Snappy o Gzip applicati a dati simili (colonne), ottenendo rapporti di compressione elevati.
- Zero-Copy Read: arraw permette di accedere ai dati direttamente dalla memoria condivisa senza dover creare muove istanze di oggetti Python
- Large Language Model Datasets: perchè le librerie come 'datasets' di Hugging Face preferiscono Arrow per gestire dataset che eccedono la capacità della RAM.

Ma cosa succede se il nostro dataset evolve nel tempo?
Evoluzione dello Schema
Gestire il cambiamento
I formati moderni gestiscono l'evoluzione dello schema, supportano la 'Schema Evolution', permettendo di aggiungere nuove colonne senza invalidare i dati storici.

- L'efficienza di archiviazione è calcolata come il rapporto tra la dimensione del dato grezzo e quella del file compresso.


In [2]:
import pandas as pd
import numpy as np
import os
import hashlib
import json

# FASE 1: GENERAZIONE DI UN DATASET AMPIO
# Per apprezzare la differenza tra i formati di file, generiamo un dataset di dimensioni significative.
# Simuliamo 1 milione di righe con dati numerici e categorici.
print("Generazione del dataset in corso...")
num_rows = 1_000_000

data = {
    'id': np.arange(num_rows),
    'valore_numerico': np.random.randn(num_rows), # Distribuzione normale
    'punteggio': np.random.randint(1, 100, size=num_rows),
    'categoria': np.random.choice(['A', 'B', 'C', 'D'], size=num_rows)
}

df = pd.DataFrame(data)

# FASE 2: CONFRONTO TRA FORMATI (CSV vs PARQUET)
# Salviamo lo stesso dataset in formato CSV (testuale) e Parquet (colonnare binario).

csv_file = "dataset_didatttico.csv"
parquet_file = "dataset_didattico.parquet"

# Salvataggio in CSV
df.to_csv(csv_file, index=False)

# Salvataggio in Parquet (formato standard per Big Data)
df.to_parquet(parquet_file, engine='pyarrow', compression='snappy')

# Calcolo e visualizzazione delle dimensioni dei file
size_csv = os.path.getsize(csv_file) / (1024 * 1024) # Dimensione in MB
size_parquet = os.path.getsize(parquet_file) / (1024 * 1024) # Dimensione in MB

print(f"\n--- CONFRONTO DIMENSIONI ---")
print(f"Dimensione file CSV:     {size_csv:.2f} MB")
print(f"Dimensione file Parquet: {size_parquet:.2f} MB")
print(f"Risparmio di spazio:     {((size_csv - size_parquet) / size_csv) * 100:.2f}%")

# FASE 3: LOGICA DI HASHING E VERSIONAMENTO (FILOSOFIA DVC)
# In ML Ops, non carichiamo i grandi file di dati su Git. Versioniamo i metadati e l'hash del file.

def calculate_md5(file_path):
    """
    Calcola l'impronta digitale (hash) del file. 
    Se anche un solo bit nel file cambia, l'hash sarà completamente diverso.
    """
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        # Leggiamo il file a blocchi per non saturare la memoria con file giganti
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

# Calcoliamo l'hash del nostro file Parquet
data_hash = calculate_md5(parquet_file)
print(f"\n--- LOGICA DVC ---")
print(f"Hash calcolato per il dataset: {data_hash}")

# Creazione del file .dvc (Metadati di puntamento)
# Questo è il file che verrebbe effettivamente salvato su Git al posto del dataset reale.
dvc_structure = {
    "outs": [
        {
            "md5": data_hash,
            "path": parquet_file,
            "size": os.path.getsize(parquet_file)
        }
    ],
    "meta": {
        "descrizione": "Dataset generato per test efficienza formati",
        "autore": "Docente",
        "versione": "1.0"
    }
}

dvc_filename = f"{parquet_file}.dvc"
with open(dvc_filename, "w") as f:
    json.dump(dvc_structure, f, indent=4)

print(f"File di versionamento creato: {dvc_filename}")
print("\nNota didattica: In un progetto reale, il file Parquet verrebbe caricato su un Cloud Storage (S3/Azure),")
print("mentre il file .dvc verrebbe tracciato da Git per garantire la riproducibilità degli esperimenti.")


Generazione del dataset in corso...

--- CONFRONTO DIMENSIONI ---
Dimensione file CSV:     30.93 MB
Dimensione file Parquet: 13.06 MB
Risparmio di spazio:     57.78%

--- LOGICA DVC ---
Hash calcolato per il dataset: e40885518931137dd1fb329611df9307
File di versionamento creato: dataset_didattico.parquet.dvc

Nota didattica: In un progetto reale, il file Parquet verrebbe caricato su un Cloud Storage (S3/Azure),
mentre il file .dvc verrebbe tracciato da Git per garantire la riproducibilità degli esperimenti.
